# RECAP FREE — Google Colab

Kaggle notebook ကို Colab အတွက် ပြင်ထားတဲ့ version ပါ။

**အရင်လုပ်ရန်:** `Runtime → Change runtime type → T4 GPU` ရွေးပါ။

Kaggle နဲ့ ကွာတာတွေ:
- `/kaggle/working` မရှိလို့ `/content` သုံးထားပြီး၊ repo ထဲက hardcode path တွေအတွက် symlink လုပ်ပေးထားတယ်
- Colab မှာ GPU တစ်လုံးပဲရလို့ NLLB ကို `cuda:0` မှာ run တယ်
- Colab မှာ `libcublas.so.12` / cuDNN 9 မပါလို့ pip နဲ့ထည့်ပြီး `LD_LIBRARY_PATH` သတ်မှတ်တယ် (**ဒီ error ကို ဖြေရှင်းတာ**)


In [ ]:
# CELL 0 — GPU စစ်မယ် + /kaggle/working symlink
import os, subprocess
print(subprocess.run(['nvidia-smi','-L'], capture_output=True, text=True).stdout or 'GPU မတွေ့ပါ! Runtime type ကို T4 GPU ပြောင်းပါ')

# repo ထဲမှာ /kaggle/working hardcode ရှိနေတာတွေ (cookies, HF cache, VoxCPM path) အတွက်
os.makedirs('/kaggle', exist_ok=True)
if not os.path.exists('/kaggle/working'):
    os.symlink('/content', '/kaggle/working')
print('OK: /kaggle/working ->', os.path.realpath('/kaggle/working'))


In [ ]:
# CELL 1 — GitHub main branch ကို clone လုပ်မယ်
from pathlib import Path
import shutil, subprocess

project = Path('/content/recap-free')
shutil.rmtree(project, ignore_errors=True)
subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/thurak538-del/Recap-ZZ.git', str(project)], check=True)
print('Project cloned:', project)
print('Latest source:', subprocess.check_output(['git', '-C', str(project), 'log', '-1', '--oneline'], text=True).strip())


In [ ]:
# CELL 2 — FFmpeg, Myanmar fonts, dependencies
%cd /content/recap-free
!apt-get update -qq
!apt-get install -y -qq ffmpeg fontconfig fonts-noto-core fonts-sil-padauk fonts-myanmar libsndfile1
!fc-cache -f
!pip install -q -r requirements-kaggle.txt
print('Dependencies ready')


In [ ]:
# CELL 2.5 — Colab fix: CUDA 12 libraries (libcublas.so.12, cuDNN 9) ထည့်မယ်
!pip install -q nvidia-cublas-cu12 "nvidia-cudnn-cu12==9.*" nvidia-cuda-runtime-cu12

import os, glob, ctypes, site

lib_dirs = []
for sp in site.getsitepackages() + [site.getusersitepackages()]:
    lib_dirs += glob.glob(os.path.join(sp, 'nvidia', '*', 'lib'))
lib_dirs = sorted(set(lib_dirs))
os.environ['LD_LIBRARY_PATH'] = ':'.join(lib_dirs + [os.environ.get('LD_LIBRARY_PATH', '')])
print('Added to LD_LIBRARY_PATH:'); [print('  ', d) for d in lib_dirs]

# ဒီ kernel ထဲမှာပါ တစ်ခါတည်း preload လုပ်ပြီး စစ်မယ်
for d in lib_dirs:
    for name in sorted(glob.glob(os.path.join(d, 'libcublas*.so.12'))) + sorted(glob.glob(os.path.join(d, 'libcudnn*.so.9'))):
        try: ctypes.CDLL(name, mode=ctypes.RTLD_GLOBAL)
        except OSError as e: print('preload fail:', name, e)
try:
    ctypes.CDLL('libcublas.so.12'); print('✅ libcublas.so.12 loaded OK')
except OSError as e:
    print('❌ still missing:', e)


In [ ]:
# CELL 3 — Local AI models download + environment (GPU တစ်လုံးအတွက်)
%cd /content/recap-free
from huggingface_hub import snapshot_download
import os
models_root = '/content/models'
voxcpm_dir = f'{models_root}/VoxCPM2'
whisper_dir = f'{models_root}/faster-whisper-large-v3'
nllb_dir = f'{models_root}/nllb-200-distilled-1.3B'
snapshot_download(repo_id='openbmb/VoxCPM2', local_dir=voxcpm_dir)
snapshot_download(repo_id='Systran/faster-whisper-large-v3', local_dir=whisper_dir)
snapshot_download(repo_id='facebook/nllb-200-distilled-1.3B', local_dir=nllb_dir)
os.environ['VOXCPM_MODEL_ID'] = voxcpm_dir
os.environ['RECAP_WHISPER_MODEL'] = whisper_dir
os.environ['RECAP_LOCAL_ONLY'] = '1'
os.environ['RECAP_NLLB_MODEL'] = nllb_dir
os.environ['RECAP_WHISPER_DEVICE'] = 'cuda'
os.environ['RECAP_WHISPER_DEVICE_INDEX'] = '0'
os.environ['RECAP_NLLB_DEVICE'] = 'cuda:0'      # Colab = GPU 1 လုံးတည်း
os.environ['RECAP_CUDA_DEVICE_COUNT'] = '1'
# VRAM မလောက်ရင် အောက်ကနှစ်ကြောင်းထဲက တစ်ကြောင်း uncomment လုပ်ပါ
# os.environ['RECAP_NLLB_DEVICE'] = 'cpu'
# os.environ['RECAP_WHISPER_COMPUTE'] = 'int8_float16'
print('All local models downloaded and paths exported.')


In [ ]:
# CELL 3.5 — Colab RAM/VRAM fix: VoxCPM မစခင် Whisper + NLLB ကို memory ကနေ ဖြုတ်မယ်
# (exit code -9 = RAM မလောက်လို့ OS က process ကို သတ်လိုက်တာ)
from pathlib import Path
p = Path('/content/recap-free/app/pipeline/tts_engine.py')
src = p.read_text(encoding='utf-8')
marker = '        ctx = mp.get_context("spawn")'
free_code = '''        # --- Colab memory fix: free Whisper/NLLB before spawning the VoxCPM worker ---
        try:
            import gc
            from app.pipeline.local_transcriber import LocalWhisperTranscriber
            from app.pipeline.local_translator import LocalNLLBTranslator
            LocalWhisperTranscriber._models.clear()
            LocalNLLBTranslator._model = None
            LocalNLLBTranslator._tokenizer = None
            LocalNLLBTranslator._loaded_path = None
            gc.collect()
            import torch
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
        except Exception as _exc:
            print(f"[memory cleanup] {_exc}", flush=True)
'''
if 'Colab memory fix' in src:
    print('already patched')
elif marker in src:
    p.write_text(src.replace(marker, free_code + marker, 1), encoding='utf-8')
    print('✅ tts_engine.py patched')
else:
    print('❌ marker not found — repo ပြောင်းသွားနိုင်ပါတယ်')

import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'


In [ ]:
# CELL 4 — server + public URL  (Cell 0 → 1 → 2 → 2.5 → 3 → 3.5 ပြီးမှ run)
%cd /content/recap-free
!python kaggle_foreground.py


## အသုံးပြုရန်

1. Cell 0 → 1 → 2 → **2.5** → 3 → **3.5** → 4 ကို အစဉ်လိုက် run ပါ။ (Cell 2.5 မှာ ✅ မပေါ်ရင် Cell 4 မ run ပါနဲ့)
2. Cell 4 ကို run လို့ ရလာတဲ့ `trycloudflare.com` link ကို ဖွင့်ပါ။
3. **Settings** ထဲမှာ API key (လိုရင်) ထည့်ပြီး AI Mode = Local AI ထားပါ။
4. မြန်မာစာတန်းထိုးအတွက် Font ကို **Noto Sans Myanmar** သို့မဟုတ် **Padauk** ရွေးပါ။
5. Error ထပ်တက်ရင် Runtime → Restart session လုပ်ပြီး အစကနေ ပြန် run ပါ။
6. Cell 4 ကို run ပြီးသားဆိုရင် Cell 3.5 ကို run ပြီး Cell 4 ကို Stop → ပြန် run ပါ (patch ဝင်ဖို့ server restart လိုတယ်)။
